# 04 Ablation Study

This notebook reproduces the logical ablation sequence used in the manuscript.

In [1]:
from pathlib import Path
import sys

repo_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(repo_root)


/root/大创/perovskite-bandgap-ml-correction


In [2]:
from src.two_step_model import ablation
ablation()


Single-step baseline (N=201): MAE=0.237 eV, R2=0.879
+ Two-step workflow (N=201): MAE=0.239 eV, R2=0.832
+ NLP augmentation (N=257): MAE=0.199 eV, R2=0.883
+ Feature engineering + bounds (N=257): MAE=0.190 eV, R2=0.890


In [3]:
import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesClassifier, ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import KFold
from imblearn.over_sampling import SMOTE

from src.data_pipeline import load_training_set, sanitize_columns, build_feature_matrix
from src.gga_filter import is_metal_by_exp

df = sanitize_columns(load_training_set(repo_root / 'data' / 'training_set_257.csv'))
df['gap_vol_ratio'] = df['band_gap'] / (df['volume'] + 1e-5)
drop_cols = ['Formula', 'E_g_Exp', 'Source', 'Priority', 'pretty_formula', 'Delta_E_g', 'is_metal_exp', 'target_delta']


In [4]:
def run_step(data, use_two_step, use_features, use_bounds):
    X, cols = build_feature_matrix(data, drop_cols)
    if not use_features and 'gap_vol_ratio' in cols:
        X = X.drop(columns=['gap_vol_ratio'])
    y = data['E_g_Exp'].values
    y_cls = is_metal_by_exp(y, threshold=0.05).astype(int)
    gga = data['band_gap'].values
    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    pred = np.zeros(len(X))

    if not use_two_step:
        reg = ExtraTreesRegressor(n_estimators=100, random_state=42)
        for train_idx, test_idx in cv.split(X):
            reg.fit(X.iloc[train_idx], y[train_idx])
            pred[test_idx] = np.clip(reg.predict(X.iloc[test_idx]), 0.0, None)
        return mean_absolute_error(y, pred), r2_score(y, pred)

    for train_idx, test_idx in cv.split(X):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train_cls = y_cls[train_idx]
        y_train_reg = y[train_idx]
        clf = ExtraTreesClassifier(n_estimators=300, max_depth=15, random_state=42)
        if use_features:
            smote = SMOTE(random_state=42)
            try:
                X_train_smote, y_train_cls_smote = smote.fit_resample(X_train, y_train_cls)
                clf.fit(X_train_smote, y_train_cls_smote)
            except Exception:
                clf.fit(X_train, y_train_cls)
        else:
            clf.fit(X_train, y_train_cls)
        pred_cls = clf.predict(X_test)
        mask_train_nm = y_train_cls == 0
        if mask_train_nm.sum() > 0:
            reg = ExtraTreesRegressor(n_estimators=500 if use_features else 200, max_features=0.5 if use_features else 1.0, max_depth=20 if use_features else None, random_state=42)
            reg.fit(X_train[mask_train_nm], y_train_reg[mask_train_nm])
            pred_reg = reg.predict(X_test)
        else:
            pred_reg = np.zeros(len(X_test))
        fold_pred = np.where(pred_cls == 1, 0.0, pred_reg)
        if use_bounds:
            nm_mask = pred_cls == 0
            fold_pred[nm_mask] = np.maximum(fold_pred[nm_mask], gga[test_idx][nm_mask] - 0.1)
        pred[test_idx] = np.clip(fold_pred, 0.0, None)
    return mean_absolute_error(y, pred), r2_score(y, pred)


In [5]:
steps = []
steps.append(('Single-step baseline (N=201)',) + run_step(df.sample(n=201, random_state=42), False, False, False))
steps.append(('+ Two-step workflow (N=201)',) + run_step(df.sample(n=201, random_state=42), True, False, False))
steps.append(('+ NLP augmentation (N=257)',) + run_step(df.copy(), True, False, False))
steps.append(('+ Feature engineering + bounds (N=257)',) + run_step(df.copy(), True, True, True))

pd.DataFrame(steps, columns=['Step', 'MAE', 'R2'])


,Step,MAE,R2
0,Single-step baseline (N=201),0.236992,0.878676
1,+ Two-step workflow (N=201),0.238904,0.831563
2,+ NLP augmentation (N=257),0.199044,0.883246
3,+ Feature engineering + bounds (N=257),0.190060,0.889799
